<a href="https://colab.research.google.com/github/be-ayush/ai-ml-learning/blob/main/HOML/HOML3_Chapter6_DimentionalityReduction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
## Finding Min Number of Dimensions for MNIST

from sklearn.datasets import fetch_openml
from sklearn.decomposition import PCA
import numpy as np

mnist = fetch_openml('mnist_784', as_frame=False)
X_train, y_train = mnist.data[:60000], mnist.target[:60000]
X_test, y_test = mnist.data[60000:], mnist.target[:60000]

pca = PCA()
pca.fit(X_train)
cumulative_sum = np.cumsum(pca.explained_variance_ratio_)
d = np.argmax(cumulative_sum >= 0.95) + 1

In [5]:
print(d)

153


In [6]:
pca = PCA(n_components = 0.95)
X_reduced = pca.fit_transform(X_train)

In [7]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import RandomizedSearchCV
from sklearn.pipeline import make_pipeline

classifier = make_pipeline(PCA(n_components = 0.95), RandomForestClassifier(random_state = 42))

param_distribution = {
    'pca__n_components': np.arange(10,80),
    'randomforestclassifier__n_estimators': np.arange(50, 500)
}

random_search = RandomizedSearchCV(classifier, param_distribution, n_iter = 10, cv = 3, verbose = 1, n_jobs = -1, random_state = 42)
random_search.fit(X_train[:1000], y_train[:1000])

Fitting 3 folds for each of 10 candidates, totalling 30 fits


RandomizedSearchCV(cv=3,
                   estimator=Pipeline(steps=[('pca', PCA(n_components=0.95)),
                                             ('randomforestclassifier',
                                              RandomForestClassifier(random_state=42))]),
                   n_jobs=-1,
                   param_distributions={'pca__n_components': array([10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26,
       27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43,
       44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 5...
       414, 415, 416, 417, 418, 419, 420, 421, 422, 423, 424, 425, 426,
       427, 428, 429, 430, 431, 432, 433, 434, 435, 436, 437, 438, 439,
       440, 441, 442, 443, 444, 445, 446, 447, 448, 449, 450, 451, 452,
       453, 454, 455, 456, 457, 458, 459, 460, 461, 462, 463, 464, 465,
       466, 467, 468, 469, 470, 471, 472, 473, 474, 475, 476, 477, 478,
       479, 480, 481, 482, 483, 484, 485, 486, 487, 488, 489, 490, 491,
       492, 493, 494, 495, 496, 497, 498, 499])},
                   random_state=42, verbose=1)

In [8]:
print(random_search.best_params_)

{'randomforestclassifier__n_estimators': np.int64(314), 'pca__n_components': np.int64(36)}


In [9]:
X_back = pca.inverse_transform(X_reduced)

In [11]:
## Incremental PCA -> When the data is too big for memory

from sklearn.decomposition import IncrementalPCA

n_batches = 100
inc_pca = IncrementalPCA(n_components = 154)
for X_batch in np.array_split(X_train, n_batches):
    inc_pca.partial_fit(X_batch)

X_reduced = inc_pca.transform(X_train)

In [12]:
filename = "my_mnist.mmap"

X_mmap = np.memmap(filename, dtype = "float32", mode = "write", shape = X_train.shape)
X_mmap[:] = X_train
X_mmap.flush()

In [13]:
X_mmap = np.memmap(filename, dtype = "float32", mode = "readonly").reshape(-1, 784)
batch_size = X_mmap.shape[0]
inc_pca = IncrementalPCA(n_components = 154, batch_size = batch_size)
inc_pca.fit(X_mmap)

IncrementalPCA(batch_size=60000, n_components=154)

In [14]:
from sklearn.random_projection import johnson_lindenstrauss_min_dim

m, e = 5000, 0.1
d = johnson_lindenstrauss_min_dim(m, eps = e)
d

np.int64(7300)